In [0]:
# ===================================================
# BLOCK 1 — SOURCE-TO-FACT RECONCILIATION (PYTHON)
# ===================================================

"""
Confirm that each core fact retains the complete accepted Silver population
at its declared grain.
"""

reconciliations = [
    (
        "lot_performance",
        "semiconplus_portfolio.silver.production_lots",
        "semiconplus_portfolio.gold.fact_lot_performance",
    ),
    (
        "unit_test_results",
        "semiconplus_portfolio.silver.unit_test_results",
        "semiconplus_portfolio.gold.fact_unit_test_results",
    ),
    (
        "equipment_events",
        "semiconplus_portfolio.silver.equipment_events",
        "semiconplus_portfolio.gold.fact_equipment_events",
    ),
]

reconciliation_results = []

for dataset_name, silver_table, fact_table in reconciliations:
    silver_count = spark.table(silver_table).count()
    fact_count = spark.table(fact_table).count()

    assert fact_count == silver_count
    reconciliation_results.append((dataset_name, silver_count, fact_count))

display(
    spark.createDataFrame(
        reconciliation_results,
        ["dataset", "silver_rows", "fact_rows"],
    )
)

print("Source-to-fact reconciliation passed.")

In [0]:
# ===================================================
# BLOCK 2 — FACT GRAIN AND SURROGATE-KEY VALIDATION (PYTHON)
# ===================================================

"""
Confirm that fact business grains and fact surrogate keys remain unique.
"""

fact_keys = {
    "semiconplus_portfolio.gold.fact_lot_performance": (
        "lot_key", "lot_id"
    ),
    "semiconplus_portfolio.gold.fact_unit_test_results": (
        "test_result_key", "test_result_id"
    ),
    "semiconplus_portfolio.gold.fact_equipment_events": (
        "equipment_event_key", "event_id"
    ),
}

from pyspark.sql import functions as F

for table_name, (surrogate_key, business_key) in fact_keys.items():
    fact_df = spark.table(table_name)

    assert fact_df.groupBy(surrogate_key).count().filter(
        F.col("count") > 1
    ).count() == 0

    assert fact_df.groupBy(business_key).count().filter(
        F.col("count") > 1
    ).count() == 0

print("Fact grain and key validation passed.")

In [0]:
%sql
-- # ===================================================
-- # BLOCK 3 — UNIT-TEST-RESULT FACT (SQL)
-- # ===================================================

-- The initial validated dataset should resolve every conformed key without
-- using an unknown dimension member.

SELECT
    SUM(CASE WHEN production_date_key = 0 THEN 1 ELSE 0 END) AS unknown_dates,
    SUM(CASE WHEN site_key = 0 THEN 1 ELSE 0 END) AS unknown_sites,
    SUM(CASE WHEN product_group_key = 0 THEN 1 ELSE 0 END) AS unknown_groups,
    SUM(CASE WHEN device_key = 0 THEN 1 ELSE 0 END) AS unknown_devices,
    SUM(CASE WHEN equipment_key = 0 THEN 1 ELSE 0 END) AS unknown_equipment
FROM semiconplus_portfolio.gold.fact_lot_performance;

-- Expected: every value is 0.

In [0]:
# ===================================================
# BLOCK 4 — LOT QUANTITY AND YIELD VALIDATION (PYTHON)
# ===================================================

"""
Independently recalculate quantity reconciliation and manufacturing yield
for every lot fact.
"""

lots_df = spark.table(
    "semiconplus_portfolio.gold.fact_lot_performance"
)

assert lots_df.filter(
    F.col("quantity_passed") + F.col("quantity_failed")
    != F.col("quantity_started")
).count() == 0

assert lots_df.filter(
    F.abs(
        F.col("manufacturing_yield").cast("double")
        - F.col("quantity_passed") / F.col("quantity_started")
    ) > 0.000001
).count() == 0

print("Lot quantity and yield validation passed.")

In [0]:
%sql
-- # ===================================================
-- # BLOCK 5 — DAILY YIELD MART RECONCILIATION (SQL)
-- # ===================================================

WITH fact_totals AS (
    SELECT
        SUM(quantity_started) AS started,
        SUM(quantity_passed) AS passed,
        SUM(quantity_failed) AS failed
    FROM semiconplus_portfolio.gold.fact_lot_performance
),
mart_totals AS (
    SELECT
        SUM(quantity_started) AS started,
        SUM(quantity_passed) AS passed,
        SUM(quantity_failed) AS failed
    FROM semiconplus_portfolio.gold.mart_daily_yield
)
SELECT
    f.started - m.started AS started_difference,
    f.passed - m.passed AS passed_difference,
    f.failed - m.failed AS failed_difference
FROM fact_totals f CROSS JOIN mart_totals m;

-- Expected: all differences are 0.



In [0]:
# ===================================================
# BLOCK 6 — DEFECT PARETO VALIDATION (PYTHON)
# ===================================================

"""
Confirm that defect counts reconcile to failed unit tests and cumulative
percentages remain within their valid range.
"""

tests_df = spark.table(
    "semiconplus_portfolio.gold.fact_unit_test_results"
)
pareto_df = spark.table(
    "semiconplus_portfolio.gold.mart_defect_pareto"
)

failed_test_count = tests_df.filter(F.col("failed_unit_count") == 1).count()
pareto_defect_count = pareto_df.agg(F.sum("defect_count")).first()[0] or 0

assert pareto_defect_count == failed_test_count
assert pareto_df.filter(
    (F.col("cumulative_defect_percentage") < 0)
    | (F.col("cumulative_defect_percentage") > 1)
).count() == 0

print("Defect Pareto validation passed.")


In [0]:
# ===================================================
# BLOCK 7 — OEE COMPONENT VALIDATION (PYTHON)
# ===================================================

"""
Confirm that Availability, Performance, Quality, and OEE remain within
the valid zero-to-one range and that OEE equals their product.
"""

oee_df = spark.table(
    "semiconplus_portfolio.gold.mart_equipment_oee_daily"
)

metric_columns = ["availability", "performance", "quality", "oee"]

for metric in metric_columns:
    assert oee_df.filter(
        F.col(metric).isNotNull()
        & ((F.col(metric) < 0) | (F.col(metric) > 1))
    ).count() == 0

assert oee_df.filter(
    F.col("oee").isNotNull()
    & (
        F.abs(
            F.col("oee").cast("double")
            - F.col("availability").cast("double")
            * F.col("performance").cast("double")
            * F.col("quality").cast("double")
        ) > 0.000005
    )
).count() == 0

print("OEE component validation passed.")

In [0]:
# ===================================================
# BLOCK 8 — LOT TRACEABILITY RECONCILIATION (PYTHON)
# ===================================================


"""
Confirm that the traceability mart retains exactly one row per accepted
production lot and that unit-test totals reconcile to the test fact.
"""

trace_df = spark.table(
    "semiconplus_portfolio.gold.mart_lot_traceability"
)

assert trace_df.count() == lots_df.count()
assert trace_df.groupBy("lot_id").count().filter(
    F.col("count") > 1
).count() == 0

assert (
    trace_df.agg(F.sum("unit_test_result_count")).first()[0]
    == tests_df.count()
)

print("Lot traceability validation passed.")

In [0]:
# ===================================================
# BLOCK 9 — CAPTURE IDEMPOTENCY BASELINE (PYTHON)
# ===================================================

"""
Capture fact and mart counts before rerunning the complete deterministic
Gold build.
"""

gold_tables = [
    "semiconplus_portfolio.gold.fact_lot_performance",
    "semiconplus_portfolio.gold.fact_unit_test_results",
    "semiconplus_portfolio.gold.fact_equipment_events",
    "semiconplus_portfolio.gold.fact_data_quality",
    "semiconplus_portfolio.gold.mart_daily_yield",
    "semiconplus_portfolio.gold.mart_device_yield",
    "semiconplus_portfolio.gold.mart_defect_pareto",
    "semiconplus_portfolio.gold.mart_equipment_oee_daily",
    "semiconplus_portfolio.gold.mart_equipment_losses",
    "semiconplus_portfolio.gold.mart_lot_traceability",
    "semiconplus_portfolio.gold.mart_data_quality_summary",
]

counts_before_rerun = {
    table_name: spark.table(table_name).count()
    for table_name in gold_tables
}

print(counts_before_rerun)

In [0]:
# ===================================================
# BLOCK 10 — IDEMPOTENCY VALIDATION (PYTHON)
# ===================================================

"""
Confirm that rebuilding the current Gold model preserves every fact and
mart population without appending duplicates.
"""

counts_after_rerun = {
    table_name: spark.table(table_name).count()
    for table_name in gold_tables
}

assert counts_after_rerun == counts_before_rerun

print("GOLD FACT AND MART IDEMPOTENCY TEST PASSED")

In [0]:
# ===================================================
# BLOCK 11 — FINAL RESULT (PYTHON)
# ===================================================

print("GOLD FACT AND ANALYTICAL MART VALIDATION PASSED")
for table_name in gold_tables:
    print(f"{table_name}: {spark.table(table_name).count():,}")